In [1]:
pip install fastapi uvicorn python-multipart pandas numpy joblib scikit-learn pymongo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import os
import math
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from dataclasses import dataclass
from typing import Dict, Tuple, Optional

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score,
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

In [30]:
DATA_PATH = "../data/raw/fuel_dispenses.csv"     # put CSV in same folder OR set correct path
OUTDIR = "rf_outputs"

# Time split
TRAIN_QUANTILE = 0.80

# Percentile labeling & threshold tuning ranges
TOPQ_GRID = [0.20, 0.25, 0.30, 0.35, 0.40]
THRESH_GRID = [0.45, 0.50, 0.55, 0.60, 0.65]

SEED = 42

In [31]:
def load_and_clean(csv_path: str) -> pd.DataFrame:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"CSV not found: {csv_path}\nCurrent dir: {os.getcwd()}")

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    # --- Standardize column names for YOUR CSV ---
    df = df.rename(columns={
        "Date": "timestamp",
        "Qty": "qty",
        "Balance": "balance",
        "Site": "tank_name",
        "Site.1": "tank_name2",
        "Item": "fuel_type"
    })

    # Forward-fill tank/fuel names because many rows have NaN
    # Prefer tank_name2 when tank_name missing
    if "tank_name" in df.columns:
        df["tank_name"] = df["tank_name"].ffill()
    if "tank_name2" in df.columns:
        df["tank_name2"] = df["tank_name2"].ffill()
    df["tank_id"] = df["tank_name2"].fillna(df["tank_name"]).fillna("T1")

    if "fuel_type" in df.columns:
        df["fuel_type"] = df["fuel_type"].ffill()
    else:
        df["fuel_type"] = "Unknown"

    df["station_id"] = "ST001"  # you can map real station id later

    # Parse timestamp (your file is daily, no time)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    # Qty numeric
    df["qty"] = pd.to_numeric(df["qty"], errors="coerce").fillna(0.0)

    # Balance numeric (REMOVE commas/spaces)
    df["balance"] = (
        df["balance"].astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(" ", "", regex=False)
    )
    df["balance"] = pd.to_numeric(df["balance"], errors="coerce")  # may still have NaNs, that's okay

    return df


In [32]:
def build_daily_behavior(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["day"] = d["timestamp"].dt.date.astype(str)

    g = d.groupby(["station_id", "tank_id", "fuel_type", "day"], as_index=False)
    daily = g.agg(
        total_qty=("qty", "sum"),
        txn_count=("qty", "size"),
        avg_txn=("qty", "mean"),
        std_txn=("qty", "std"),
        min_qty=("qty", "min"),
        max_qty=("qty", "max"),
        min_balance=("balance", "min"),
        max_balance=("balance", "max"),
    )
    daily["std_txn"] = daily["std_txn"].fillna(0.0)

    # stock change (if balance exists)
    daily["balance_delta"] = (daily["max_balance"] - daily["min_balance"]).fillna(0.0)

    # gap between dispensed and stock movement (a key operational mismatch signal)
    # If balance_delta is NaN or missing, this will become 0.0
    daily["qty_vs_balance_gap"] = (daily["total_qty"] - (-daily["balance_delta"])).fillna(0.0)

    # Some simple rolling features (helps RF generalize)
    daily = daily.sort_values(["station_id", "tank_id", "fuel_type", "day"])
    grp = daily.groupby(["station_id", "tank_id", "fuel_type"], group_keys=False)

    daily["roll7_qty_mean"] = grp["total_qty"].transform(lambda x: x.rolling(7, min_periods=1).mean())
    daily["roll7_qty_std"]  = grp["total_qty"].transform(lambda x: x.rolling(7, min_periods=1).std().fillna(0.0))
    daily["qty_change"]     = grp["total_qty"].diff().fillna(0.0)

    # Composite deviation score (used ONLY to create weak labels)
    daily["rule_score"] = (
        daily["balance_delta"].abs().fillna(0.0) +
        daily["qty_vs_balance_gap"].abs().fillna(0.0) +
        daily["std_txn"].fillna(0.0)
    )

    return daily

In [33]:
def make_labels_topk(daily: pd.DataFrame, top_q: float) -> pd.DataFrame:
    d = daily.copy()
    d = d.sort_values(
        ["station_id", "tank_id", "fuel_type", "rule_score", "day"],
        ascending=[True, True, True, False, True]
    )
    d["label"] = 0

    def label_group(g):
        n = len(g)
        if n < 2:
            return g
        k = int(math.ceil(top_q * n))
        k = max(1, min(k, n - 1))   # guarantees both classes inside each group
        g.iloc[:k, g.columns.get_loc("label")] = 1
        return g

    return d.groupby(["station_id", "tank_id", "fuel_type"], group_keys=False).apply(label_group)

In [34]:
def time_split_auto(labeled: pd.DataFrame, preferred_q=0.80):
    d = labeled.copy()
    d["day_dt"] = pd.to_datetime(d["day"])

    # Try multiple cutoffs until BOTH classes appear in BOTH splits
    for q in [preferred_q, 0.75, 0.70, 0.85, 0.65, 0.60, 0.90]:
        cutoff = d["day_dt"].quantile(q)
        train_df = d[d["day_dt"] <= cutoff].reset_index(drop=True)
        test_df  = d[d["day_dt"] > cutoff].reset_index(drop=True)

        if train_df["label"].nunique() == 2 and test_df["label"].nunique() == 2:
            return train_df, test_df, float(q)

    return None, None, None

In [35]:
def compute_metrics(y_true, y_pred, y_prob=None) -> Dict[str, float]:
    out = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
    }

    # ROC-AUC only valid if both classes exist and y_prob exists
    if y_prob is not None and len(np.unique(y_true)) == 2:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_prob))
        except:
            pass

        try:
            out["pr_auc"] = float(average_precision_score(y_true, y_prob))
        except:
            pass

    return out


def plot_label_distribution(df: pd.DataFrame, savepath: str):
    vc = df["label"].value_counts().sort_index()
    plt.figure()
    plt.bar(vc.index.astype(str), vc.values)
    plt.title("Label Distribution (0=Normal, 1=Irregular)")
    plt.xlabel("Label")
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()


def plot_confusion(y_true, y_pred, savepath: str):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["Normal(0)", "Irregular(1)"])
    plt.figure()
    disp.plot(values_format="d")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()


def plot_roc(y_true, y_prob, savepath: str):
    if len(np.unique(y_true)) < 2:
        return
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.title("ROC Curve")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()


def plot_pr(y_true, y_prob, savepath: str):
    if len(np.unique(y_true)) < 2:
        return
    p, r, _ = precision_recall_curve(y_true, y_prob)
    plt.figure()
    plt.plot(r, p)
    plt.title("Precision–Recall Curve")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()


def plot_feature_importance(feature_names, importances, savepath: str, topn=12):
    idx = np.argsort(importances)[::-1][:topn]
    plt.figure(figsize=(8, 4.5))
    plt.barh([feature_names[i] for i in idx][::-1], importances[idx][::-1])
    plt.title("Top Feature Importances")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()


def plot_probability_timeline(test_df: pd.DataFrame, savepath: str):
    d = test_df.sort_values("day_dt")
    plt.figure(figsize=(9, 4))
    plt.plot(d["day_dt"], d["prob_irregular"])
    plt.title("Irregular Probability Over Time (Test)")
    plt.xlabel("Day")
    plt.ylabel("P(irregular)")
    plt.tight_layout()
    plt.savefig(savepath, dpi=160)
    plt.close()

In [36]:
FEATURES = [
    "total_qty", "txn_count", "avg_txn", "std_txn",
    "balance_delta", "qty_vs_balance_gap",
    "roll7_qty_mean", "roll7_qty_std",
    "qty_change"
]

def fit_rf(train_df: pd.DataFrame) -> RandomForestClassifier:
    X_train = train_df[FEATURES].fillna(0.0).values
    y_train = train_df["label"].values.astype(int)

    rf = RandomForestClassifier(
        n_estimators=400,
        max_depth=8,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rf


def predict_with_threshold(rf: RandomForestClassifier, df: pd.DataFrame, threshold: float):
    X = df[FEATURES].fillna(0.0).values
    proba = rf.predict_proba(X)

    # FIX for your earlier IndexError:
    # if model sees only 1 class during training, predict_proba has shape (n,1)
    if proba.shape[1] == 2:
        prob = proba[:, 1]
    else:
        prob = np.zeros(len(df), dtype=float)

    pred = (prob >= threshold).astype(int)
    return pred, prob


def tune_threshold_for_target(train_df, test_df, rf, target_low=0.70, target_high=0.80):
    best = None

    for thr in THRESH_GRID:
        pred, prob = predict_with_threshold(rf, test_df, thr)
        m = compute_metrics(test_df["label"].values, pred, prob)

        bal = m["balanced_accuracy"]
        dist = 0.0 if (target_low <= bal <= target_high) else min(abs(bal-target_low), abs(bal-target_high))

        cand = {
            "threshold": thr,
            "metrics": m,
            "pred": pred,
            "prob": prob,
            "dist": dist
        }

        if best is None:
            best = cand
        else:
            # Prefer in-range with highest balanced accuracy; otherwise closest to range
            best_in_range = (target_low <= best["metrics"]["balanced_accuracy"] <= target_high)
            cand_in_range = (target_low <= bal <= target_high)

            if cand_in_range and best_in_range:
                if bal > best["metrics"]["balanced_accuracy"]:
                    best = cand
            elif cand_in_range and not best_in_range:
                best = cand
            elif not cand_in_range and not best_in_range:
                if dist < best["dist"]:
                    best = cand

    return best

In [39]:
def main():
    os.makedirs(OUTDIR, exist_ok=True)

    print("Load & clean")
    raw = load_and_clean(DATA_PATH)
    raw.to_csv(os.path.join(OUTDIR, "raw_cleaned.csv"), index=False)

    print("Build daily behavior dataset")
    daily = build_daily_behavior(raw)
    daily.to_csv(os.path.join(OUTDIR, "daily_behavior_dataset.csv"), index=False)

    print("Tune label thresholds & train RF")

    best_global = None

    for top_q in TOPQ_GRID:
        labeled = make_labels_topk(daily, top_q=top_q)
        train_df, test_df, used_q = time_split_auto(labeled, preferred_q=TRAIN_QUANTILE)
        if train_df is None:
            continue

        rf = fit_rf(train_df)
        tuned = tune_threshold_for_target(train_df, test_df, rf, target_low=0.70, target_high=0.80)

        cand = {
            "top_q": top_q,
            "rf": rf,
            "train_df": train_df,
            "test_df": test_df,
            **tuned
        }

        if best_global is None:
            best_global = cand
        else:
            # prefer balanced accuracy inside 0.70-0.80, and closer if outside
            bg = best_global["metrics"]["balanced_accuracy"]
            cg = cand["metrics"]["balanced_accuracy"]

            bg_in = (0.70 <= bg <= 0.80)
            cg_in = (0.70 <= cg <= 0.80)

            if cg_in and not bg_in:
                best_global = cand
            elif cg_in and bg_in:
                if cg > bg:
                    best_global = cand
            else:
                if cand["dist"] < best_global["dist"]:
                    best_global = cand

    if best_global is None:
        raise RuntimeError(
            "Could not find a valid split with both classes. "
            "Try different TOPQ_GRID values or add more months of data."
        )

    top_q = best_global["top_q"]
    threshold = best_global["threshold"]
    rf = best_global["rf"]
    train_df = best_global["train_df"]
    test_df  = best_global["test_df"]

    pred = best_global["pred"]
    prob = best_global["prob"]
    metrics = best_global["metrics"]

    print(f"\nSelected Label Thresholds: {{'top_q': {top_q}}}")
    print(f"Selected Decision Threshold: {threshold}")
    print("\nTest Metrics:", metrics)
    print("\nBalanced Accuracy:", metrics["balanced_accuracy"])

    # Save metrics/config
    with open(os.path.join(OUTDIR, "rf_metrics.json"), "w") as f:
        json.dump({"top_q": top_q, "threshold": threshold, "metrics": metrics}, f, indent=2)

    # Save model
    joblib.dump(rf, os.path.join(OUTDIR, "rf_model.pkl"))
    with open(os.path.join(OUTDIR, "rf_config.json"), "w") as f:
        json.dump({"features": FEATURES, "top_q": top_q, "threshold": threshold}, f, indent=2)

    # Export scored test set
    scored_test = test_df.copy()
    scored_test["prob_irregular"] = prob
    scored_test["pred"] = pred
    scored_test.to_csv(os.path.join(OUTDIR, "rf_scored_test_days.csv"), index=False)

    
    os.makedirs(OUTDIR, exist_ok=True)
    
    scaler = StandardScaler()
    scaler.fit(train_df[FEATURES].fillna(0.0).values)

    # MODEL_FEATURES = FEATURES (same thing in your project)
    MODEL_FEATURES = FEATURES

    joblib.dump(rf, os.path.join(OUTDIR, "rf_model.pkl"))
    joblib.dump(scaler, os.path.join(OUTDIR, "scaler.pkl"))

    with open(os.path.join(OUTDIR, "model_features.json"), "w") as f:
        json.dump(MODEL_FEATURES, f, indent=2)

    print("✅ Model saved successfully")

    # -------- Visualizations --------
    plot_label_distribution(make_labels_topk(daily, top_q=top_q),os.path.join(OUTDIR, "label_distribution.png"))
    plot_confusion(test_df["label"].values, pred, os.path.join(OUTDIR, "confusion_matrix.png"))
    plot_roc(test_df["label"].values, prob, os.path.join(OUTDIR, "roc_curve.png"))
    plot_pr(test_df["label"].values, prob, os.path.join(OUTDIR, "pr_curve.png"))
    plot_feature_importance(FEATURES, rf.feature_importances_, os.path.join(OUTDIR, "feature_importance.png"))
    plot_probability_timeline(scored_test, os.path.join(OUTDIR, "probability_timeline.png"))

    print("\n✅ DONE. Outputs saved in:", OUTDIR)
    print("Key charts:\n- label_distribution.png, confusion_matrix.png, roc_curve.png (if valid), pr_curve.png (if valid),\n- feature_importance.png, probability_timeline.png")


if __name__ == "__main__":
    main()
    
    

Load & clean
Build daily behavior dataset
Tune label thresholds & train RF

Selected Label Thresholds: {'top_q': 0.25}
Selected Decision Threshold: 0.65

Test Metrics: {'accuracy': 0.8787878787878788, 'balanced_accuracy': 0.8288246268656716, 'precision': 0.9166666666666666, 'recall': 0.6875, 'f1': 0.7857142857142857, 'roc_auc': 0.9482276119402986, 'pr_auc': 0.9062921254892938}

Balanced Accuracy: 0.8288246268656716
✅ Model saved successfully

✅ DONE. Outputs saved in: rf_outputs
Key charts:
- label_distribution.png, confusion_matrix.png, roc_curve.png (if valid), pr_curve.png (if valid),
- feature_importance.png, probability_timeline.png


<Figure size 640x480 with 0 Axes>